# 03 · DML nas tabelas Delta do bucket `bronze`

Reproduz, sobre as tabelas Delta que agora estão no **MinIO**, as operações de DML do Trabalho 1:
**INSERT**, **UPDATE** e **DELETE** (e também **MERGE**), além de **HISTORY** e **TIME TRAVEL**.
No fim, uma célula de **verificações** confere automaticamente os resultados.

| Operação | Exemplo (o mesmo do Trabalho 1) |
|---|---|
| INSERT | novo aluno Felipe (id 301), um check-in e um pagamento dele |
| UPDATE | reajuste de 10% no plano Mensal; Ana troca para o plano Anual |
| DELETE | Diego (id 4) cancela: saem os check-ins, os pagamentos e o cadastro |
| MERGE  | Carla muda de cidade (atualiza) e Gabriela se matricula (insere) |

**Antes de rodar:** notebook `02` executado. Para rodar este notebook de novo, rode o `02` antes (ele recarrega o bronze).

## 1. Configuração e SparkSession

> **Primeira execução:** o Spark baixa da internet os pacotes `.jar` do Delta Lake e do conector S3A
> (incluindo o SDK da AWS, com cerca de 280 MB). Pode levar alguns minutos. Depois disso fica em cache em `~/.ivy2`.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Raiz do projeto (funciona abrindo o notebook pelo VS Code, pelo JupyterLab ou pelo nbconvert)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (ROOT / ".env").exists():
    raise FileNotFoundError("Arquivo .env não encontrado. Na raiz do projeto rode: cp .env.example .env")
load_dotenv(ROOT / ".env", override=True)

MINIO_ENDPOINT   = os.getenv("MINIO_ENDPOINT")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")
LANDING_BUCKET   = os.getenv("MINIO_LANDING_BUCKET")
BRONZE_BUCKET    = os.getenv("MINIO_BRONZE_BUCKET")

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("DML_Delta")
    .master("local[*]")
    # Delta Lake + conector S3A (para o Spark ler e gravar no MinIO como se fosse o Amazon S3)
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.warehouse.dir", str(ROOT / "spark-warehouse"))
    .config("spark.driver.memory", "2g")
    # MinIO via S3A
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark

## 2. Registrar as tabelas do bronze no Spark SQL

Os dados já estão no MinIO; aqui só **registramos** cada pasta como uma tabela no catálogo do Spark, apontando para ela
com `LOCATION`. Isso cria tabelas **não gerenciadas** (externas). A diferença para as gerenciadas é o assunto do notebook 04.

In [ ]:
from delta.tables import DeltaTable

TABELAS = ["unidades", "planos", "alunos", "checkins", "pagamentos"]
caminho = {t: f"s3a://{BRONZE_BUCKET}/{t}" for t in TABELAS}

spark.sql("CREATE DATABASE IF NOT EXISTS academia")
for t in TABELAS:
    spark.sql(f"CREATE TABLE IF NOT EXISTS academia.{t} USING delta LOCATION '{caminho[t]}'")

spark.sql("SHOW TABLES IN academia").show()

In [ ]:
# Guardamos a versão atual de cada tabela ANTES dos DML (usada no time travel e nas verificações)
versao_inicial = {t: DeltaTable.forPath(spark, caminho[t]).history(1).first()["version"] for t in TABELAS}
print("Versão de cada tabela antes dos DML:", versao_inicial)

ja_aplicado = spark.sql("SELECT COUNT(*) FROM academia.alunos WHERE id_aluno IN (301, 302)").first()[0]
if ja_aplicado:
    raise RuntimeError("Os DML deste notebook já foram aplicados. Rode o notebook 02 de novo (recarrega o bronze) "
                       "e depois rode este.")

In [ ]:
print("Registros por tabela antes dos DML:")
for t in TABELAS:
    print(f"  {t:<11} {spark.table(f'academia.{t}').count():>6}")

---
## 3. INSERT

### 3.1 INSERT com SQL — novo aluno

In [ ]:
spark.sql("""
    INSERT INTO academia.alunos VALUES
      (301, 'Felipe Costa', 'felipe@email.com', 'Criciúma', 2, DATE '2024-06-10', true)
""")
spark.sql("SELECT * FROM academia.alunos WHERE id_aluno = 301").show()

### 3.2 INSERT com DataFrame — primeiro check-in do Felipe (tabela `checkins`)

In [ ]:
from datetime import datetime

proximo_id = spark.sql("SELECT MAX(id_checkin) FROM academia.checkins").first()[0] + 1
novo_checkin = spark.createDataFrame(
    [(proximo_id, 301, 3, datetime(2024, 6, 11, 7, 30))],
    schema="id_checkin INT, id_aluno INT, id_unidade INT, data_hora TIMESTAMP",
)
novo_checkin.write.format("delta").mode("append").save(caminho["checkins"])

spark.sql("SELECT * FROM academia.checkins WHERE id_aluno = 301").show()

### 3.3 INSERT ... SELECT — primeiro pagamento do Felipe (tabela nova `pagamentos`)

In [ ]:
spark.sql("""
    INSERT INTO academia.pagamentos
    SELECT MAX(id_pagamento) + 1, 301, DATE '2024-06-01', DATE '2024-06-10', 89.90, 'PIX', 'PAGO'
    FROM academia.pagamentos
""")
spark.sql("SELECT * FROM academia.pagamentos WHERE id_aluno = 301").show()

---
## 4. UPDATE

### 4.1 UPDATE com SQL — reajuste de 10% no plano Mensal

In [ ]:
print("ANTES:")
spark.sql("SELECT * FROM academia.planos ORDER BY id_plano").show()

spark.sql("""
    UPDATE academia.planos
    SET valor_mensal = ROUND(valor_mensal * 1.10, 2)
    WHERE nome = 'Mensal'
""").show()

print("DEPOIS:")
spark.sql("SELECT * FROM academia.planos ORDER BY id_plano").show()

### 4.2 UPDATE com a API `DeltaTable` — Ana troca o plano Mensal pelo Anual

In [ ]:
from pyspark.sql.functions import lit

DeltaTable.forPath(spark, caminho["alunos"]).update(
    condition="id_aluno = 1",
    set={"id_plano": lit(3)},
)
spark.sql("""
    SELECT a.id_aluno, a.nome, a.id_plano, p.nome AS plano
    FROM academia.alunos a JOIN academia.planos p ON p.id_plano = a.id_plano
    WHERE a.id_aluno = 1
""").show()

---
## 5. DELETE — o aluno Diego (id 4) cancelou

Primeiro saem os registros que dependem dele (check-ins e pagamentos) e depois o cadastro.
No SQL Server essa ordem seria **obrigatória** por causa das FKs; no Delta ela é uma **boa prática**,
porque o Delta não verifica chaves estrangeiras.

In [ ]:
for tabela in ["checkins", "pagamentos", "alunos"]:
    antes = spark.sql(f"SELECT COUNT(*) FROM academia.{tabela} WHERE id_aluno = 4").first()[0]
    spark.sql(f"DELETE FROM academia.{tabela} WHERE id_aluno = 4")
    depois = spark.sql(f"SELECT COUNT(*) FROM academia.{tabela} WHERE id_aluno = 4").first()[0]
    print(f"  {tabela:<11} linhas do Diego: {antes} -> {depois}")

---
## 6. MERGE (INSERT + UPDATE num só comando)

Chega um lote com a Carla (que mudou de cidade) e a Gabriela (aluna nova). Quem já existe é atualizado; quem não existe é inserido.

In [ ]:
from datetime import date

lote = spark.createDataFrame(
    [
        (3,   "Carla Mendes",   "carla@email.com",    "Içara",    2, date(2024, 3, 12), True),
        (302, "Gabriela Alves", "gabriela@email.com", "Criciúma", 1, date(2024, 6, 15), True),
    ],
    schema="id_aluno INT, nome STRING, email STRING, cidade STRING, id_plano INT, data_matricula DATE, ativo BOOLEAN",
)
lote.createOrReplaceTempView("lote_alunos")

spark.sql("""
    MERGE INTO academia.alunos AS destino
    USING lote_alunos AS origem
    ON destino.id_aluno = origem.id_aluno
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""").show()

spark.sql("SELECT * FROM academia.alunos WHERE id_aluno IN (1, 2, 3, 4, 5, 301, 302) ORDER BY id_aluno").show()

---
## 7. HISTORY — o log de transações guarda cada operação

In [ ]:
spark.sql("DESCRIBE HISTORY academia.alunos") \
    .select("version", "timestamp", "operation", "operationMetrics") \
    .orderBy("version") \
    .show(truncate=60)

---
## 8. TIME TRAVEL — consultar a tabela como era antes dos DML

Usamos a versão guardada no início do notebook (antes de qualquer DML): o Diego volta a aparecer,
a Ana está no plano 1, a Carla em Criciúma e o Felipe e a Gabriela ainda não existem.

In [ ]:
v0 = versao_inicial["alunos"]
print(f"alunos na versão {v0} (antes dos DML):")
spark.sql(f"""
    SELECT * FROM academia.alunos VERSION AS OF {v0}
    WHERE id_aluno IN (1, 3, 4, 301, 302) ORDER BY id_aluno
""").show()

print("alunos na versão atual:")
spark.sql("SELECT * FROM academia.alunos WHERE id_aluno IN (1, 3, 4, 301, 302) ORDER BY id_aluno").show()

---
## 9. Verificações automáticas

Se qualquer resultado não for o esperado, a célula falha com `AssertionError` e diz qual verificação quebrou.

In [ ]:
from decimal import Decimal

def valor(sql):
    return spark.sql(sql).first()[0]

checks = {
    "plano Mensal reajustado para 109.89":
        valor("SELECT valor_mensal FROM academia.planos WHERE id_plano = 1") == Decimal("109.89"),
    "Ana (1) no plano Anual (3)":
        valor("SELECT id_plano FROM academia.alunos WHERE id_aluno = 1") == 3,
    "Carla (3) agora em Içara":
        valor("SELECT cidade FROM academia.alunos WHERE id_aluno = 3") == "Içara",
    "Felipe (301) e Gabriela (302) inseridos":
        valor("SELECT COUNT(*) FROM academia.alunos WHERE id_aluno IN (301, 302)") == 2,
    "Diego (4) removido de alunos, checkins e pagamentos":
        sum(valor(f"SELECT COUNT(*) FROM academia.{t} WHERE id_aluno = 4") for t in ["alunos", "checkins", "pagamentos"]) == 0,
    "alunos: 300 + 2 inseridos - 1 removido = 301":
        valor("SELECT COUNT(*) FROM academia.alunos") == 301,
    "time travel: versão inicial tem 300 alunos, com o Diego":
        valor(f"SELECT COUNT(*) FROM academia.alunos VERSION AS OF {versao_inicial['alunos']}") == 300
        and valor(f"SELECT COUNT(*) FROM academia.alunos VERSION AS OF {versao_inicial['alunos']} WHERE id_aluno = 4") == 1,
}

for descricao, ok in checks.items():
    print(("  OK    " if ok else "  FALHOU ") + descricao)
assert all(checks.values()), "Alguma verificação falhou (veja acima)."
print("\nTodas as verificações passaram.")

In [ ]:
spark.stop()
print("SparkSession encerrada.")